In [6]:
import pandas as pd
import geopandas as gpd
import numpy as np
import pyogrio
import os

In [8]:
folder = r"C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara"

### Zonas con IDs estandarizados

In [16]:
zones_standarized = gpd.read_file(os.path.join(folder, "red_shapefiles", "zones_agebs_visum", "zonas_agebs_standarized.shp"))
zones_standarized = zones_standarized.rename(columns={"id_mun_age":"id_mun_ageb"})

# Map of ageb to visum_id (id_mun_ageb)
ageb_to_visum_id = dict(zip(zones_standarized['clave_ageb'], zones_standarized['id_mun_ageb']))

In [11]:
zones_standarized.dtypes

clave_ageb       object
clave_enti        int64
clave_muni        int64
clave_loca        int64
ageb             object
nombre_mun       object
tipo_ageb        object
poblacion_        int64
area_m2         float64
area_km2        float64
establecim        int64
empleados_      float64
densidad_p      float64
densidad_e      float64
densidad_1      float64
distancia_      float64
consecutiv        int64
id_mun_ageb       int64
geometry       geometry
dtype: object

In [17]:
zones_standarized[zones_standarized['clave_ageb']== '140440011']

,clave_ageb,clave_enti,clave_muni,clave_loca,ageb,nombre_mun,tipo_ageb,poblacion_,area_m2,area_km2,establecim,empleados_,densidad_p,densidad_e,densidad_1,distancia_,consecutiv,id_mun_ageb,geometry
473,140440011,14,44,0,0011,Ixtlahuacán de los Membrillos,rural,15789,9.175450e+07,91.754504,13,2725.438533,172.078746,0.141682,29.703594,32.82126,31,1031,"POLYGON ((2374526.729 941684.611, 2374586.477 ..."


#### Matriz OD de Demanda (que preparo Daniel)

In [27]:
matriz_od_long = pd.read_excel(os.path.join(folder, "Matriz Origen Destino", "matriz_put_ap.xlsx"))

# Limpiar string values in Origen and Destino columns
# remove leading and trailing spaces and convert to string type
matriz_od_long['Origen'] = matriz_od_long['Origen'].astype(str).str.strip()
matriz_od_long['Destino'] = matriz_od_long['Destino'].astype(str).str.strip()

# Replace '99999000A' with '14097059A' in Origen and Destino columns
matriz_od_long['Origen'] = matriz_od_long['Origen'].replace('99999000A', '14097059A')
matriz_od_long['Destino'] = matriz_od_long['Destino'].replace('99999000A', '14097059A')

# Map columns Origen & Destino to visum_id for insertion of matrix into visum
matriz_od_long['Origen_map'] = matriz_od_long['Origen'].map(ageb_to_visum_id)
matriz_od_long['Destino_map'] = matriz_od_long['Destino'].map(ageb_to_visum_id)

print(f"Total of {matriz_od_long['Ponderador'].sum():,} trips in the matrix por hora")
matriz_od_long

Total of 721,596 trips in the matrix por hora


,Unnamed: 0,Modo,Hora_inicio,Origen,Destino,Ponderador,Origen_map,Destino_map
0,112,Transporte Público,4,140440011,1403900011170,15,1031.0,96.0
1,113,Transporte Público,4,140440011,1412000015374,15,1031.0,8315.0
2,114,Transporte Público,4,140970602,1409700201013,118,5242.0,5079.0
3,115,Transporte Público,4,140972416,1412000010636,53,5387.0,8015.0
4,116,Transporte Público,4,140980243,1409800011928,9,6237.0,6099.0
...,...,...,...,...,...,...,...,...
10339,10451,Transporte Público,8,141200758080A,1412000010710,33,8539.0,8021.0
10340,10452,Transporte Público,8,141200758080A,1412002313823,33,8539.0,8469.0
10341,10453,Transporte Público,8,141240001018A,1403900011378,20,9006.0,113.0
10342,10454,Transporte Público,8,141240001018A,1403900011626,21,9006.0,133.0


In [28]:
# Viajes validos entre las 2,203 zonas
od_zonas_validas = matriz_od_long[
    matriz_od_long['Origen_map'].notna() & matriz_od_long['Destino_map'].notna()
].copy()
od_zonas_validas[['Origen_map', 'Destino_map']] = od_zonas_validas[['Origen_map', 'Destino_map']].astype(int)

# Viajes con accesos carreteros (9999...) que aun no tienen correspondencia de zona
# quedan pendientes para ser asignados a la matriz de visum demanda od
od_zonas_pendientes = matriz_od_long[
    matriz_od_long['Origen_map'].isna() | matriz_od_long['Destino_map'].isna()
].copy()

print(f"Viajes válidos entre las 2,203 zonas: {len(od_zonas_validas):,}")
print(f"Viajes con accesos carreteros que aun no tienen correspondencia de zona: {len(od_zonas_pendientes):,}")

Viajes válidos entre las 2,203 zonas: 10,208
Viajes con accesos carreteros que aun no tienen correspondencia de zona: 136


In [29]:
od_zonas_validas

,Unnamed: 0,Modo,Hora_inicio,Origen,Destino,Ponderador,Origen_map,Destino_map
0,112,Transporte Público,4,140440011,1403900011170,15,1031,96
1,113,Transporte Público,4,140440011,1412000015374,15,1031,8315
2,114,Transporte Público,4,140970602,1409700201013,118,5242,5079
3,115,Transporte Público,4,140972416,1412000010636,53,5387,8015
4,116,Transporte Público,4,140980243,1409800011928,9,6237,6099
...,...,...,...,...,...,...,...,...
10339,10451,Transporte Público,8,141200758080A,1412000010710,33,8539,8021
10340,10452,Transporte Público,8,141200758080A,1412002313823,33,8539,8469
10341,10453,Transporte Público,8,141240001018A,1403900011378,20,9006,113
10342,10454,Transporte Público,8,141240001018A,1403900011626,21,9006,133


### Leer Zonas de Visum (2,203)

In [21]:
import win32com.client as com

#Red base GDL (con 2,203 zonas)
red_base = os.path.join(folder, "Red Base GDL", "RedBase Conectores y Atts", "RedBase 150826.ver")
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(red_base)
C = com.constants

In [30]:
zone_values = Visum.Net.Zones.GetMultiAttValues("No")

zone_nos = np.array(
    [int(value) for _, value in zone_values],
    dtype=np.int64
)

print(f"Número de zonas en Visum: {len(zone_nos):,}")

Número de zonas en Visum: 2,203


In [31]:
zone_to_index = {
    zone_no: index
    for index, zone_no in enumerate(zone_nos)
}

# array en ceros
n_zones = len(zone_nos)
demand_array = np.zeros((n_zones, n_zones), dtype=np.float64)

# indices de origen y destino en la matriz
origin_indices = (
    od_zonas_validas
    ["Origen_map"]
    .map(zone_to_index)
    .to_numpy(dtype=np.int64)
)

destination_indices = (
    od_zonas_validas["Destino_map"]
    .map(zone_to_index)
    .to_numpy(dtype=np.int64)
)

# Numero de viajes
demand_values = (
    od_zonas_validas["Ponderador"]
    .to_numpy(dtype=np.float64)
)

# Llenar el array con los viajes reales
np.add.at(
    demand_array,
    (origin_indices, destination_indices),
    demand_values
)

demand_array

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [32]:
print("Dimensión:", demand_array.shape)
print(f"Demanda total CSV: {od_zonas_validas['Ponderador'].sum():,.3f}")
print(f"Demanda total matriz: {demand_array.sum():,.3f}")
print(f"Pares con demanda: {np.count_nonzero(demand_array):,}")

Dimensión: (2203, 2203)
Demanda total CSV: 717,255.000
Demanda total matriz: 717,255.000
Pares con demanda: 9,309


### Get Visum matrix to fill

In [33]:
matrix_no = 4

visum_matrix = Visum.Net.Matrices.ItemByKey(matrix_no)
print("Matrix No:", visum_matrix.AttValue("No"))
print("Name:", visum_matrix.AttValue("Name"))

Matrix No: 4.0
Name: Bus OD AP


In [34]:
visum_matrix.SetValues(
    demand_array,
    Add=False
)